<a href="https://colab.research.google.com/github/chenwh0/Big-Data-Model-management-project/blob/main/data_understanding_and_light_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Install & Imports**

In [5]:
import pandas
import numpy
from sklearn.cluster import DBSCAN

# **Data preprocessing & cleaning**

In [6]:
FOLDER = "./"
FILENAMES = ["waterqualityMO2015to2025.csv", "waterqualityMO2005to2015.csv", "waterqualityMO2000to2005.csv"]

In [13]:
dataframe = pandas.concat((pandas.read_csv(FOLDER + filename, low_memory=False) for filename in FILENAMES), ignore_index=True)

## Get rid of duplicates

In [14]:
dataframe["CleanedActivityStartDate"] = pandas.to_datetime(
    dataframe["ActivityStartDate"] + " " + dataframe["ActivityStartTime/Time"].fillna(""),
    format="mixed",
    errors="coerce"
)
dataframe.drop_duplicates(
    subset=[
        "MonitoringLocationIdentifier",
        "CleanedActivityStartDate",
        "CharacteristicName",
        "ResultMeasureValue"
    ], inplace=True
)

## Get rid of rows without longitude & latitude

In [15]:
print("Rows without latitude =", dataframe["ActivityLocation/LatitudeMeasure"].isna().sum(), "| Rows without latitude = ", dataframe["ActivityLocation/LatitudeMeasure"].isna().sum())
dataframe.dropna(subset=["ActivityLocation/LatitudeMeasure"], inplace=True)
print("After dropping...")
print("Rows without latitude =", dataframe["ActivityLocation/LatitudeMeasure"].isna().sum(), "| Rows without latitude = ", dataframe["ActivityLocation/LatitudeMeasure"].isna().sum())

Rows without latitude = 9070 | Rows without latitude =  9070
After dropping...
Rows without latitude = 0 | Rows without latitude =  0


## Show dataframe info

In [16]:
dataframe.reset_index(drop=True, inplace=True)

print(dataframe.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1217024 entries, 0 to 1217023
Data columns (total 82 columns):
 #   Column                                             Non-Null Count    Dtype         
---  ------                                             --------------    -----         
 0   OrganizationIdentifier                             1217024 non-null  object        
 1   OrganizationFormalName                             1217024 non-null  object        
 2   ActivityIdentifier                                 1217024 non-null  object        
 3   ActivityTypeCode                                   1217024 non-null  object        
 4   ActivityMediaName                                  1217024 non-null  object        
 5   ActivityMediaSubdivisionName                       9085 non-null     object        
 6   ActivityStartDate                                  1217024 non-null  object        
 7   ActivityStartTime/Time                             287896 non-null   object      

In [17]:
characteristics_counts = dataframe["CharacteristicName"].value_counts()
print("What data was being collected the most?")
print(f"Top 10 {characteristics_counts[:10]}")

What data was being collected the most?
Top 10 CharacteristicName
Specific conductance      127112
Flow                      109982
Temperature, water         68239
Nitrogen                   54314
Phosphorus                 52369
Dissolved oxygen (DO)      48478
pH                         39355
Chlorophyll a              37037
Chlorophyll                31249
Total suspended solids     30896
Name: count, dtype: int64


# **DBSCAN clustering by Latitude & Longitude**

* Note: Unable to run DBSCAN on full dataframe because of resource restrictions

In [ ]:
coords = dataframe[["ActivityLocation/LatitudeMeasure", "ActivityLocation/LongitudeMeasure"]].to_numpy()
coords_rad = numpy.radians(coords) # Convert to radians (required for haversine)

# DBSCAN with haversine metric
kms_per_radian = 6371.0088
epsilon = 15 / kms_per_radian  # 15 km radius

database = DBSCAN(eps=epsilon, min_samples=5, metric='haversine')
dataframe["cluster"] = database.fit_predict(coords_rad)

## Example

In [ ]:
print(f"Total clusters = {dataframe["cluster"].nunique()}")
cluster_counts = dataframe["cluster"].value_counts()
print("Which cluster has most points?")
print(f"Top 10 {cluster_counts[:10]}")
print("\nClusters distribution:")
print(f"{cluster_counts.describe()}")

Total clusters = 105
Which cluster has most points?
Top 10 cluster
8     37683
1     34793
2     25250
7     13215
3      7036
4      6484
16     4999
28     4458
20     3414
11     3274
Name: count, dtype: int64

Clusters distribution:
count      105.000000
mean      1783.857143
std       5655.334050
min          8.000000
25%         52.000000
50%        300.000000
75%       1044.000000
max      37683.000000
Name: count, dtype: float64


In [ ]:
valid_clusters = cluster_counts[cluster_counts > cluster_counts.describe()["25%"]].index
dataframe_filtered = dataframe[dataframe["cluster"].isin(valid_clusters)]  # Filter out geographical clusters containing less than 25% percentile # of points
cluster_counts = dataframe_filtered["cluster"].value_counts()
print(cluster_counts)

cluster
8     37683
1     34793
2     25250
7     13215
3      7036
      ...  
71       80
92       80
58       73
40       64
61       57
Name: count, Length: 76, dtype: int64


In [ ]:
for clusterID in cluster_counts.index:
    print(f"Cluster {clusterID} top collected data")
    print(dataframe_filtered[dataframe_filtered["cluster"] == clusterID]["CharacteristicName"].value_counts()[:5])
    print()

Cluster 8
CharacteristicName
Nitrogen                  2930
Phosphorus                2182
Temperature, water        2097
Total suspended solids    1747
Chlorophyll a             1534
Name: count, dtype: int64

Cluster 1
CharacteristicName
Nitrogen                  2676
Temperature, water        1960
Phosphorus                1936
Total suspended solids    1637
Chlorophyll a             1273
Name: count, dtype: int64

Cluster 2
CharacteristicName
Nitrogen              2005
Temperature, water    1486
Phosphorus            1409
Chlorophyll a         1111
Depth                  866
Name: count, dtype: int64

Cluster 7
CharacteristicName
Nitrogen              1080
Phosphorus             778
Temperature, water     751
Chlorophyll a          713
Depth                  631
Name: count, dtype: int64

Cluster 3
CharacteristicName
Nitrogen                  534
Temperature, water        434
Phosphorus                431
Escherichia coli          329
Total suspended solids    310
Name: count, dtyp